<a href="https://colab.research.google.com/github/Precytess/AIPND-revision/blob/master/Human_Activity_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from tensorflow import keras
from tensorflow.keras import Sequential ,layers
from kerastuner.tuners import RandomSearch
from keras.layers import Dense, Dropout
import tensorflow as tf


from sklearn.metrics import accuracy_score, roc_curve, roc_auc_score

In [ ]:
test = pd.read_csv("/content/drive/MyDrive/DATA SCIENCE AND ML/Human Activity Detection using smartphone/dataset/test.csv")
train = pd.read_csv("/content/drive/MyDrive/DATA SCIENCE AND ML/Human Activity Detection using smartphone/dataset/train.csv")

#DATA EXPLORATION

In [ ]:
print(f'training set size: {train.shape}')
print(f'testing set size: {test.shape}\n')

In [ ]:
train.tail()

In [ ]:
test.tail()

In [ ]:
train['Activity'].value_counts()

In [ ]:
test['Activity'	].value_counts()

The data is pretty evenly distributed

In [ ]:
print(train.duplicated().sum())
print(test.duplicated().sum())

In [ ]:
train.replace([np.inf, -np.inf], np.nan, inplace = True)

In [ ]:
test.replace([np.inf, -np.inf], np.nan, inplace = True)

In [ ]:
print(train.isna().sum())
print(test.isna().sum())

In [ ]:
plt.title('datapoints per activity')
sns.countplot(train['Activity'])
plt.xticks(rotation = 90)

plt.show()

In [ ]:
X_train = train.drop('Activity', axis = 1)
y_train = train['Activity']
X_test = test.drop('Activity', axis = 1)
y_test = test['Activity']


In [ ]:
rf = RandomForestClassifier(random_state = 42)

In [ ]:
rf.fit(X_train, y_train)

In [ ]:
train_pred = rf.predict(X_train)
y_pred = rf.predict(X_test)

train_acc = accuracy_score(y_train, train_pred )
acc = accuracy_score(y_test, y_pred)
print(train_acc)
acc

In [ ]:
df = pd.concat([train, test])
df = train.sample(frac = 1)
df.index.duplicated().any()

In [ ]:
X = df.drop(['Activity', 'subject'], axis = 1)
y = df['Activity']

In [ ]:
tsne = TSNE(n_components=2, perplexity=30.0, n_iter=1000, random_state=42)

In [ ]:
tsne_data = tsne.fit_transform(X)

In [ ]:


tsne_data.shape

In [ ]:
df = pd.DataFrame({'x':tsne_data[:,0], 'y':tsne_data[:,1] ,'label':y})
sns.lmplot(data=df, x='x', y='y', hue='label', fit_reg=False, \
                   palette="Set1",markers=['^','v','s','o', '1','2'])
plt.title("TSNE VISUALIZATION")
plt.savefig('TSNE plot')
plt.show()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
pca = PCA()
pca.fit(X_scaled)
explained_variance = pca.explained_variance_ratio_

plt.figure(figsize=(8, 5))
plt.plot(np.cumsum(explained_variance))
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Explained Variance vs. Number of Components')
plt.show()


In [ ]:
n_components = 100
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_scaled)

# Variance explained by each principal component
print('Explained variance by component: ', pca.explained_variance_ratio_)

In [ ]:
pca_df = pd.DataFrame(data=X_pca, columns=[f'PC{i+1}' for i in range(n_components)])

In [ ]:
pca_df.shape

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x='PC1', y='PC2', data=pca_df, hue= y, palette='viridis')
plt.title('PCA Projection onto the first 2 principal components')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
le = LabelEncoder()
y = le.fit_transform(y)

In [ ]:
print(pca_df.shape)
print(y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(pca_df, y, test_size = 0.3,shuffle = True , random_state = 8598)

In [ ]:
rf.fit(X_train, y_train)

In [ ]:
y_pred = rf.predict(X_test)

In [ ]:
acc = accuracy_score(y_pred, y_test)
acc

In [ ]:
y_prob = rf.predict_proba(X_test)

#auc = round(roc_auc_score(y_test, y_prob), 4)
#plt.plot(fpr,tpr,label="Random Forest Classifier, AUC="+str(auc))

#from sklearn.preprocessing import LabelBinarizer

#label_binarizer = LabelBinarizer().fit(y_train)
#y_onehot_test = label_binarizer.transform(y_test)
#y_onehot_test.shape  # (n_samples, n_classes)


In [ ]:
#from sklearn.metrics import RocCurveDisplay
#from itertools import cycle


#fpr = dict()
#tpr = dict()
#roc_auc = dict()
#for i in range(len(df.)):
    #fpr[i], tpr[i], _ = roc_curve(y_test == iris.target_names[i], y_score[:, i])
    #roc_auc[i] = auc(fpr[i], tpr[i])

In [ ]:
#visualizing correlation
corr= df.corr()
sns.heatmap(corr)

In [ ]:
#Removing highly correlated columns
#def correlation(df, threshold):
#  col_corr = set()
 # for i in range(len(corr.columns)):
 #   for j in range(i):
  #    if abs(corr.iloc[i,j]) > threshold:
  #      col_name = corr.columns[i]
  #      col_corr.add(col_name)
 # return col_corr

In [ ]:
#corr_features = correlation(df, 0.80)
#print(f'dropping {len(corr_features)} features')

#df = df.drop(corr_features, axis = 1)

In [ ]:
model = Sequential()
model.add(Dense(units=64,kernel_initializer='normal',activation='sigmoid',input_dim=X_train.shape[1]))
model.add(Dropout(0.2))
model.add(Dense(units=6,kernel_initializer='normal',activation='softmax'))
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
history = model.fit(X_train, y_train, batch_size = 64, epochs= 10,validation_data = (X_test,y_test))

In [ ]:
model.predict(X_test)

In [ ]:
def build_model(hp):
    model = keras.Sequential()
    for i in range(hp.Int('num_layers', 2, 25)):
        model.add(layers.Dense(units = hp.Int('units' + str(i), min_value=32, max_value=512, step=32),
                               kernel_initializer= hp.Choice('initializer', ['uniform', 'normal']),
                               activation= hp.Choice('activation', ['relu', 'sigmoid', 'tanh'])))
    model.add(layers.Dense(6, kernel_initializer= hp.Choice('initializer', ['uniform', 'normal']), activation='softmax'))
    model.add(
            Dropout(0.2))
    model.compile(
        optimizer = 'adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'])
    return model


tuner = RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials= 5,
    executions_per_trial=3,
    directory='project', project_name = 'Human_activity_recognition')

tuner.search_space_summary()

In [ ]:
tuner.search(X_train, y_train,
             epochs= 10,
             validation_data=(X_test, y_test))

In [ ]:
tuner.results_summary

In [ ]:
model=tuner.get_best_models(num_models=1)[0]
history = model.fit(X_train,y_train, epochs=51, validation_data=(X_test,y_test))

In [ ]:
model.summary()

Callback = tf.keras.callbacks.EarlyStopping(monitor='accuracy', patience=3)
mo_fitt = model.fit(X_train,y_train, epochs=200, validation_data=(X_test,y_test), callbacks=Callback)

In [ ]:
accuracy = mo_fitt.history['accuracy']
loss = mo_fitt.history['loss']
validation_loss = mo_fitt.history['val_loss']
validation_accuracy = mo_fitt.history['val_accuracy']


plt.figure(figsize=(15, 7))
plt.subplot(2, 2, 1)
plt.plot(range(5), accuracy, label='Training Accuracy')
plt.plot(range(5), validation_accuracy, label='Validation Accuracy')
plt.legend(loc='upper left')
plt.title('Accuracy : Training Vs Validation ')



plt.subplot(2, 2, 2)
plt.plot(range(5), loss, label='Training Loss')
plt.plot(range(5), validation_loss, label='Validation Loss')
plt.title('Loss : Training Vs Validation ')
plt.legend(loc='upper right')
plt.show()
